# Cheatsheet: Multiple Variable Linear Regression for Economics

A quick-reference companion to the lab. Formulas, NumPy syntax, economic interpretation tips, common pitfalls, and a "top 10 applications" list — all in one place.


## 1. The Model

$$ f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w}\cdot\mathbf{x} + b = w_0x_0+w_1x_1+\cdots+w_{n-1}x_{n-1}+b $$

| Symbol | Economic meaning (general) | Shape |
|---|---|---|
| $\mathbf{X}$ | data matrix: rows = observations (firms/households/countries/periods), columns = explanatory variables | (m, n) |
| $\mathbf{y}$ | outcome variable (spending, wage, GDP growth, price, ...) | (m,) |
| $\mathbf{w}$ | marginal effects — "holding everything else fixed" coefficients | (n,) |
| $b$ | intercept — predicted outcome when all features are 0 | scalar |
| $m$ | number of observations | int |
| $n$ | number of explanatory variables (features) | int |


## 2. Cost Function (Mean Squared Error, halved)

$$ J(\mathbf{w},b) = \frac{1}{2m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)}\right)^2 $$

- Convex in $(\mathbf{w},b)$ for linear regression &rarr; gradient descent will find the global minimum (no local-minima worries).
- The `1/2` is a convention that cancels a factor of 2 when differentiating — it doesn't change *where* the minimum is.


## 3. Gradients

$$ \frac{\partial J}{\partial w_j} = \frac{1}{m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)}\right)x_j^{(i)} \qquad
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)}\right) $$

**Update rule** (simultaneous update!):
$$ w_j \leftarrow w_j - \alpha\frac{\partial J}{\partial w_j}, \qquad b \leftarrow b - \alpha\frac{\partial J}{\partial b} $$


## 4. NumPy Syntax Quick Reference


In [ ]:
import numpy as np

X = np.array([[4360, 5, 16, 45],
              [2950, 3, 12, 40],
              [1560, 2,  9, 35]])
y = np.array([3210, 2100, 1250])
w = np.array([0.31, 129.42, 124.31, -17.61])
b = -0.28

# --- Prediction for ONE example (vectorized) ---
pred_single = np.dot(X[0], w) + b          # scalar

# --- Prediction for ALL examples at once (fully vectorized, no loop at all) ---
pred_all = X @ w + b                       # shape (m,)   '@' is matrix multiply

# --- Cost, fully vectorized (no loops) ---
m = X.shape[0]
errors = (X @ w + b) - y                   # shape (m,)
cost_vectorized = np.sum(errors ** 2) / (2 * m)

# --- Gradient, fully vectorized (no loops) ---
dj_dw_vectorized = (X.T @ errors) / m      # shape (n,)
dj_db_vectorized = np.sum(errors) / m      # scalar

print("pred_single       :", pred_single)
print("pred_all           :", pred_all)
print("cost_vectorized     :", cost_vectorized)
print("dj_dw_vectorized  :", dj_dw_vectorized)
print("dj_db_vectorized  :", dj_db_vectorized)

> **Tip:** The loop-based implementations in the lab exist for pedagogical clarity. In practice (and for speed on real datasets with many rows/columns), always prefer the fully vectorized versions above: `X @ w + b`, `X.T @ errors`, etc. They are mathematically identical.


## 5. Feature Scaling (why & how)

Economic variables routinely differ by orders of magnitude (income in \$10,000s vs. years of education in single digits vs. interest rates as decimals like 0.03). This causes:
- Gradient descent to need a **tiny learning rate** to avoid diverging on the large-scale feature
- Painfully **slow convergence** on the small-scale features
- Coefficients that are **hard to compare** in magnitude (a big $w$ doesn't mean "more important" if its feature's scale is tiny)

**Z-score normalization** (most common):
$$ x_j^{norm} = \frac{x_j - \mu_j}{\sigma_j} $$


In [ ]:
def zscore_normalize(X):
    '''
    Normalize each column (feature) of X to zero mean, unit variance.
    Returns X_norm, plus mu and sigma so new/test data can be scaled the same way.
    '''
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

X_norm, mu, sigma = zscore_normalize(X)
print("mu   :", mu)
print("sigma:", sigma)
print("X_norm:\n", X_norm)

# IMPORTANT: apply the SAME mu/sigma (computed on training data) to any new example
def predict_from_raw(x_raw, w_norm, b_norm, mu, sigma):
    x_norm = (x_raw - mu) / sigma
    return np.dot(x_norm, w_norm) + b_norm

## 6. Model Evaluation Metrics (beyond cost)

| Metric | Formula | Interpretation |
|---|---|---|
| MAE | $\frac{1}{m}\sum\lvert \hat y_i - y_i\rvert$ | average absolute error, same units as $y$ |
| MSE | $\frac{1}{m}\sum(\hat y_i-y_i)^2$ | penalizes big misses more |
| RMSE | $\sqrt{MSE}$ | back in the same units as $y$ |
| $R^2$ | $1-\dfrac{\sum(\hat y_i-y_i)^2}{\sum(y_i-\bar y)^2}$ | share of variance in $y$ explained by the model (1.0 = perfect) |


In [ ]:
def evaluate(y_true, y_pred):
    err = y_pred - y_true
    mae = np.mean(np.abs(err))
    mse = np.mean(err**2)
    rmse = np.sqrt(mse)
    ss_res = np.sum(err**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    r2 = 1 - ss_res/ss_tot
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

y_pred_demo = X @ w + b
print(evaluate(y, y_pred_demo))

## 7. Economic Interpretation Cheatsheet

| If your model predicts... | $w_j$ on feature $j$ is called / interpreted as... |
|---|---|
| Consumption ~ Income | **Marginal Propensity to Consume (MPC)** |
| Wage ~ Years of education | **Return to schooling** (often estimated in log-wage form &rarr; % return) |
| Quantity demanded ~ Price | (negative) **slope of demand**; in log-log form, an **elasticity** |
| GDP growth ~ Investment, trade openness, inflation | **growth-accounting coefficients** |
| Housing price ~ sq.ft., bedrooms, location dummies | **hedonic price** for each attribute |

**Log-log trick for elasticities:** if you regress $\ln y$ on $\ln x_j$, the coefficient $w_j$ is directly interpretable as an **elasticity**: "a 1% increase in $x_j$ is associated with a $w_j$% change in $y$." This is extremely common in economics (demand elasticities, labor supply elasticities, trade elasticities).


## 8. Common Pitfalls & Sanity Checks

1. **Forgetting to scale features** &rarr; gradient descent diverges or crawls. *Fix:* z-score normalize; always store `mu`/`sigma` to reuse at prediction time.
2. **Multicollinearity** — two features that move together (e.g., income & years of education) make individual coefficients unstable/hard to interpret, even if overall prediction is fine. *Check:* correlation matrix `np.corrcoef(X.T)`, or Variance Inflation Factor (VIF).
3. **Confusing correlation with causation** — a positive $w_j$ does not prove $x_j$ *causes* $y$ to rise. Omitted-variable bias is everywhere in observational economic data.
4. **Overfitting with too many features relative to observations** — with $m$ close to $n$, you can drive cost to ~0 without learning anything generalizable (exactly what happens with `m=3, n=4` in this lab!). *Fix:* more data, regularization (Ridge/Lasso), or fewer features.
5. **Extrapolating outside the observed range** — a model fit on households earning \$1,500–\$4,500/month should not be trusted to predict spending for a household earning \$500,000/month.
6. **Not simultaneously updating $w$ and $b$** in gradient descent — always compute *all* gradients using the *current* parameters before updating any of them.
7. **Ignoring the intercept's (lack of) meaning** — $b$ is "predicted $y$ when every $x_j=0$," which is often a nonsensical hypothetical (e.g., a household with zero income, zero size, zero age).
8. **Learning rate too large** &rarr; cost oscillates or explodes to `nan`/`inf`. Too small &rarr; painfully slow convergence. *Fix:* try multiples of 3 (…0.001, 0.003, 0.01, 0.03…) and watch the cost-vs-iteration curve.


## 9. Gradient Descent Debug Checklist

- [ ] Does cost **decrease every iteration**? If not, lower `alpha`.
- [ ] Have you **scaled features**? (Check `X.std(axis=0)` — wildly different numbers is a red flag.)
- [ ] Are `w` and `b` updated **simultaneously** (using gradients computed from the *same* old parameters)?
- [ ] Does `compute_cost(X, y, w_final, b_final)` roughly match the final value of `J_history`?
- [ ] Do predictions on a **holdout set** (not used for fitting) look reasonable, not just training predictions?


## 10. Top 10 Applications of Multiple-Variable Linear Regression in Economics

1. **Consumption function estimation** — predicting household/aggregate spending from income, wealth, interest rates, demographics (this lab's example).
2. **Wage / earnings (Mincer) equations** — wages explained by education, experience, experience², occupation, region — the workhorse model of labor economics.
3. **Demand estimation** — quantity demanded as a function of own price, competitors' prices, income, advertising (often in log-log form for elasticities).
4. **Hedonic pricing models** — housing or asset prices explained by their attributes (size, location, amenities) to back out the implicit price of each characteristic.
5. **Production functions & productivity** — output explained by capital, labor, materials, technology proxies (e.g., Cobb-Douglas in log form is linear regression).
6. **GDP growth / macro forecasting** — growth explained by investment rate, trade openness, inflation, human capital, institutional quality (cross-country growth regressions).
7. **Cost-of-capital / asset pricing** — expected stock/portfolio returns explained by multiple risk factors (e.g., Fama-French 3/5-factor models are literally multiple linear regressions).
8. **Inflation / Phillips-curve modeling** — inflation explained by unemployment gap, expected inflation, import prices, output gap.
9. **Policy impact / program evaluation (as a control-variable tool)** — estimating a treatment effect (e.g., minimum wage, tax credit) while controlling for other observable determinants of the outcome.
10. **Real estate / regional economics** — rent or property values explained by local income, population growth, interest rates, and zoning/regulatory variables, used by urban and regional economists and by mortgage/appraisal models.

*(Almost every one of these historically began as an OLS multiple regression; modern applied work adds fixed effects, instrumental variables, or richer functional forms — but the linear-regression mechanics in this lab are the shared foundation.)*


## 11. From-Scratch vs. Libraries — Quick Comparison

| | This lab (from scratch) | `numpy.linalg.lstsq` | `scikit-learn LinearRegression` | `statsmodels OLS` |
|---|---|---|---|---|
| Shows the mechanics (gradient descent) | ✅ | ❌ (closed-form) | ❌ | ❌ (closed-form) |
| Good for teaching | ✅ | Partial | ❌ | ❌ |
| Gives p-values / standard errors | ❌ | ❌ | ❌ | ✅ |
| Production-ready / fast on big data | ❌ | ✅ | ✅ | ✅ |
| One-line fit | ❌ | ✅ | ✅ | ✅ |

```python
# scikit-learn equivalent, for reference
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(X_train, y_train)
print(model.coef_, model.intercept_)

# statsmodels equivalent (adds p-values, R^2, confidence intervals — most used in applied economics)
import statsmodels.api as sm
X_with_const = sm.add_constant(X_train)
result = sm.OLS(y_train, X_with_const).fit()
print(result.summary())
```
